# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Qualifying Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

qualifying_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("race_name", StringType(), True),
    StructField("circuit_id", StringType(), True),
    StructField("number", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("driver_id", StringType(), False),
    StructField("code", StringType(), True),
    StructField("given_name", StringType(), True),
    StructField("family_name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("constructor_id", StringType(), True),
    StructField("constructor_name", StringType(), True),
    StructField("q1", StringType(), True),
    StructField("q2", StringType(), True),
    StructField("q3", StringType(), True),
])

qualifying_input_path = f"{processed_folder_path}/qualifying/csv/qualifying.csv"

qualifying_df = spark.read \
    .option("header", True) \
    .schema(qualifying_schema) \
    .csv(qualifying_input_path)


# 3) Transform Qualifying Data:

The steps included:

- Drop column "url", "code", "given_name", "family_name", "family_name", "nationality", "constructor_name".
- Create Surrogate Key.
- Add Data Source and File Date.
- Fill Null cells with "None".

In [0]:
from pyspark.sql.functions import lit

qualifying_with_audit_df = qualifying_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

qualifying_date_df = add_ingestion_date(qualifying_with_audit_df)
qualifying_fill_df = qualifying_date_df.fillna("None")
qualifying_dropped_df = qualifying_fill_df.drop("url", "code", "given_name", "family_name", "family_name", "nationality", "constructor_name")

qualifying_final_df = add_surrogate_key(
    qualifying_dropped_df,
    key_column_name="qualifying_sk",
    hash_columns=["season", "round", "race_name", "circuit_id", "number", "position", "driver_id", 
                  "constructor_id", "q1", "q2", "q3"],
)

print("Final columns going into the write:", qualifying_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
qualifying_output_path = f"{processed_folder_path}/qualifying/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=qualifying_final_df,
    db_name="f1_processed",
    table_name="qualifying",
    output_path=qualifying_output_path,
    merge_key_columns=["season", "round"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(qualifying_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/qualifying/delta",
    presentation_directory=f"{presentation_folder_path}/fact_qualifying/delta",
    db_name="f1_presentation",
    table_name="fact_qualifying",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_qualifying/delta"))

# 5) Save backup Qualifying in CSV format:

In [0]:
import io
import csv

qualifying_backup_path = f"{presentation_folder_path}/fact_qualifying/csv/fact_qualifying.csv"

backup_rows = [row.asDict() for row in qualifying_final_df.collect()]
backup_fieldnames = qualifying_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(qualifying_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {qualifying_backup_path}")